In [23]:
import pandas as pd
import numpy as np

In [24]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [25]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
    "fuel_efficiency_mpg"
]

target = "fuel_efficiency_mpg"

In [26]:
df = df[features]

In [27]:
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [28]:
#Q1

In [29]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [30]:
# Q2

In [31]:
median = df.horsepower.median()
median

np.float64(254.0)

In [32]:
# Prepare and split the dataset

In [33]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

del df_train[target]
del df_val[target]
del df_test[target]

In [34]:
def prepare_X(df, fill=0):
    df_num = df.fillna(fill)
    X = df_num.values
    return X

In [35]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones , X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [36]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [37]:
# Q3

In [38]:
X_train_zero = prepare_X(df_train)
w0, w = train_linear_regression(X_train_zero, y_train)

X_val_zero = prepare_X(df_val)
y_pred = w0 + X_val_zero.dot(w)

rmse_zero = rmse(y_val, y_pred)
round(rmse_zero, 3)

np.float64(0.071)

In [39]:
horsepower_mean = df_train["horsepower"].mean()
X_train_mean = prepare_X(df_train,horsepower_mean)
w0, w = train_linear_regression(X_train_mean, y_train)

X_val_mean = prepare_X(df_val)
y_pred = w0 + X_val_mean.dot(w)

rmse_mean = rmse(y_val, y_pred)
round(rmse_mean, 3)

np.float64(0.072)

In [40]:
# Q4

In [41]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones , X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]
    

In [42]:
X_train = prepare_X(df_train)
X_val = prepare_X(df_val)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r
    )

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    print(round(score, 4))

0.0715
0.0715
0.0717
0.0727
0.0733
0.0734
0.0735


In [43]:
#Q5

In [44]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    
    y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
    y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
    y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

    X_train = prepare_X(df_train)
    X_val = prepare_X(df_val)

    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r=0
    )

    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    scores.append(score)

std = np.std(scores)
print(round(std, 3))


0.0
